In [21]:
%pip install numpy pandas transformers torch openpyxl


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [22]:
"""
===============================================================================
AQ098-3-3 Robo-Based Advisor | FINBERT (LEXICON & RULE-BASED ALGO TRADE)
Same-day news -> same-day entry -> multi-horizon forward performance
===============================================================================
PART 1 = EDITABLE settings. PART 2 = ENGINE (mechanics, avoid editing).
PART 3 = RUN.

STRATEGY: only headlines published DURING Malaysian trading hours (09:00-17:00 MYT by default) are used, and only to trade THAT SAME DAY.
News is never carried over to trade the next day. All same-day headlines are averaged into one net daily signal (avoids counting repeat headlines
as separate trades); if the signal is net positive, go long at the first tradable bar after the news, same day. If there's no tradable bar left
that day, the signal is simply skipped, not rolled forward.

For every trade taken, the return is measured at NINE forward horizons:
1, 2, 3, 4, 5, 10, 20 calendar days later. No stop-loss/take-profit in this version.

First run needs internet: FinBERT model weights (~400MB) download once from huggingface.co, then are cached locally.
===============================================================================
"""

from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =============================================================================
# PART 1 -- EDITABLE
# =============================================================================
NEWS_CSV_PATH   = "US stocks headlines.csv"
PRICE_PATH      = "MI_Technovation_Cleaned.xlsx"     # .xlsx or .csv; see load_price()
OUTPUT_DIR      = Path.cwd()

"""FINBERT Configuration and Parameter Justification"""
FINBERT_MODEL_NAME     = "ProsusAI/finbert" # Which pretrained sentiment model to download and use for scoring headlines.
FINBERT_BATCH_SIZE     = 16 # How many headlines get scored at once. Bigger batches run faster but use more memory; 16 is a safe default.

MIN_HEADLINE_CHARS = 12 # Any headline shorter than 12 characters gets thrown out before scoring, too short to be a real headline.
MIN_ALPHA_RATIO    = 0.55 # At least 55% of headline's characters must be letters/spaces. Filter out rows that are mostly numbers/symbols/garbage text.

"""
Only news published in this MYT window is used, and only to trade THAT # SAME DAY (see engine note above). Set to your own market's session hours.
"""

TRADING_HOURS_START = "09:00:00"
TRADING_HOURS_END   = "17:00:00"

"""
MINIMUM DAILY SCORE: All of a day's headlines get averaged into one net sentiment score (positive probability - negative probability). 
That score must be above 0.0 to trigger a trade. 0.0 means "any net lean toward positive, however small" qualifies. 
Raise this to demand more conviction before trading. """

MIN_DAILY_SCORE = 0.0

""" Holding days"""
HORIZONS_DAYS = [1, 2, 3, 4, 5, 10, 20]   # calendar days forward from entry

""" Backtesting Assumptions and Parameters """
INITIAL_CAPITAL      = 10_000.0
POSITION_SIZE_PCT    = 1.00
TRANSACTION_COST_PCT = 0.5     # round-trip brokerage, percent of notional

""" Risk-free rate"""
RISK_FREE_RATE_ANNUAL = 0.03

"""
Breakeven stop: once price has risen enough to cover entry_price AND the round-trip brokerage cost (i.e. the trade could be closed for >=0% net),
the stop arms at that breakeven level. If price then falls back to it, the trade exits there instead of being allowed to round-trip into an
actual loss. Has no effect if price never reaches breakeven in the first place -- it only protects gains already covering cost, it isn't a
downside stop from entry.
"""

USE_BREAKEVEN_STOP = True


# =============================================================================
# PART 2 -- ENGINE
# =============================================================================
def load_news(csv_path: str) -> pd.DataFrame:
    """news CSV has separate date (D/M/YYYY) + time (H:MM:SS, occasionally
    hour=24 meaning next day) columns, both MYT. Combines into one datetime."""
    df = pd.read_csv(csv_path)

    def split_time(t):
        hh, mm, ss = t.strip().split(":")
        extra_days, hh = divmod(int(hh), 24)
        return f"{hh:02d}:{mm}:{ss}", extra_days

    fixed = df["time"].apply(split_time)
    date = pd.to_datetime(df["date"], dayfirst=True) + pd.to_timedelta(fixed.apply(lambda x: x[1]), unit="D")
    df["datetime_myt"] = pd.to_datetime(date.dt.strftime("%Y-%m-%d") + " " + fixed.apply(lambda x: x[0]))
    return df.drop(columns=["date", "time"]).sort_values("datetime_myt").reset_index(drop=True)


def filter_trading_hours(df, start_hhmmss, end_hhmmss):
    clock = df["datetime_myt"].dt.time
    keep = (clock >= pd.to_datetime(start_hhmmss).time()) & (clock <= pd.to_datetime(end_hhmmss).time())
    return df.loc[keep].reset_index(drop=True)


def filter_headline_quality(df, min_chars, min_alpha_ratio):
    def alpha_ratio(s):
        s = s or ""
        return sum(c.isalpha() or c.isspace() for c in s) / len(s) if s else 0.0
    keep = (df["headline_raw"].str.len().fillna(0) >= min_chars) & \
           (df["headline_raw"].fillna("").apply(alpha_ratio) >= min_alpha_ratio)
    return df.loc[keep].reset_index(drop=True)


def score_sentiment_finbert(df, model_name, batch_size):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).eval()
    id2label = {k: v.lower() for k, v in model.config.id2label.items()}

    prob_pos, prob_neg = [], []
    headlines = df["headline_raw"].tolist()
    with torch.no_grad():
        for i in range(0, len(headlines), batch_size):
            enc = tokenizer(headlines[i:i + batch_size], return_tensors="pt",
                             padding=True, truncation=True, max_length=64)
            probs = torch.nn.functional.softmax(model(**enc).logits, dim=-1)
            for p in probs:
                pmap = {id2label[j]: float(p[j]) for j in id2label}
                prob_pos.append(pmap.get("positive", 0.0))
                prob_neg.append(pmap.get("negative", 0.0))

    scored = df.copy()
    scored["prob_positive"] = prob_pos
    scored["prob_negative"] = prob_neg
    scored["score"] = scored["prob_positive"] - scored["prob_negative"]   # -1..+1
    return scored


def aggregate_daily_signal(news, min_daily_score):
    """All same-day headlines -> one net score for that calendar day, so a
    day with 10 headlines about the same story isn't 10 separate trades."""
    daily = (news.assign(entry_date=news["datetime_myt"].dt.normalize())
                  .groupby("entry_date")
                  .agg(daily_score=("score", "mean"), n_headlines=("score", "size"))
                  .reset_index())
    daily["go_long"] = daily["daily_score"] > min_daily_score
    return daily


def load_price(path: str) -> pd.DataFrame:
    """Reads .xlsx (Exchange Time/Close columns) or a plain time/close .csv.
    Forward-fills any blank prices from illiquid minutes rather than
    dropping them."""
    if str(path).lower().endswith(".xlsx"):
        px = pd.read_excel(path).rename(columns={"Exchange Time": "time", "Close": "close"})[["time", "close"]]
    else:
        px = pd.read_csv(path, parse_dates=["time"])[["time", "close"]]
        if px["time"].dt.tz is not None:
            px["time"] = px["time"].dt.tz_localize(None)
    px = px.sort_values("time").reset_index(drop=True)
    px["close"] = px["close"].ffill()
    return px.dropna(subset=["close"]).reset_index(drop=True)


def _first_bar_same_day_after(price, ts):
    """First tradable bar at/after ts, but ONLY within the same calendar
    day -- returns (None, None) if the news lands with no bars left that
    day, rather than rolling forward to the next session."""
    day_end = ts.normalize() + pd.Timedelta(days=1)
    window = price[(price["time"] >= ts) & (price["time"] < day_end)]
    return (None, None) if window.empty else (window.iloc[0]["time"], window.iloc[0]["close"])


def _first_bar_at_or_after(price, ts):
    window = price[price["time"] >= ts]
    return (None, None) if window.empty else (window.iloc[0]["time"], window.iloc[0]["close"])


def _breakeven_stop_time(price, entry_time, entry_price, txn_cost_pct, max_days):
    """Scans forward from entry. Once close >= breakeven_price for the
    first time, the stop is armed; the first time close <= breakeven_price
    AFTER that, the trade is stopped out there. Returns (stop_time,
    breakeven_price) or (None, breakeven_price) if never triggered."""
    breakeven_price = entry_price * (1 + txn_cost_pct / 100.0)
    window = price[(price["time"] > entry_time) & (price["time"] <= entry_time + pd.Timedelta(days=max_days))]
    armed = False
    for _, row in window.iterrows():
        if not armed and row["close"] >= breakeven_price:
            armed = True
        elif armed and row["close"] <= breakeven_price:
            return row["time"], breakeven_price
    return None, breakeven_price


def simulate_trades(daily_signal, price, horizons_days, txn_cost_pct, use_breakeven_stop):
    max_h = max(horizons_days)
    rows = []
    for _, r in daily_signal.iterrows():
        if not r["go_long"]:
            continue

        entry_time, entry_price = _first_bar_same_day_after(price, r["entry_date"])
        if entry_time is None:
            continue   # no bars left that day -- skipped, never carried to next day

        stop_time, breakeven_price = (_breakeven_stop_time(price, entry_time, entry_price, txn_cost_pct, max_h)
                                       if use_breakeven_stop else (None, None))

        rec = {"entry_date": r["entry_date"], "n_headlines": r["n_headlines"],
               "daily_score": r["daily_score"], "entry_time": entry_time, "entry_price": entry_price,
               "breakeven_price": breakeven_price, "stopped_out": stop_time is not None,
               "stop_time": stop_time}

        for h in horizons_days:
            horizon_time = entry_time + pd.Timedelta(days=h)
            if stop_time is not None and stop_time <= horizon_time:
                # already stopped out at breakeven by this horizon -- flat at ~0% net
                rec[f"exit_time_{h}d"] = stop_time
                rec[f"return_{h}d"] = (breakeven_price - entry_price) / entry_price - txn_cost_pct / 100.0
                continue
            exit_time, exit_price = _first_bar_at_or_after(price, horizon_time)
            if exit_time is None:
                rec[f"exit_time_{h}d"], rec[f"return_{h}d"] = pd.NaT, np.nan
                continue
            net_ret = (exit_price - entry_price) / entry_price - txn_cost_pct / 100.0
            rec[f"exit_time_{h}d"], rec[f"return_{h}d"] = exit_time, net_ret

        rows.append(rec)
    return pd.DataFrame(rows)


def performance_report(trades, horizons_days, capital, size_pct, rf_annual):
    out = []
    for h in horizons_days:
        r = trades[f"return_{h}d"].dropna()
        if len(r) == 0:
            out.append({"days": h, "trades": 0})
            continue

        equity = (1 + r * size_pct).cumprod() * capital
        drawdown = (equity - equity.cummax()) / equity.cummax()
        periods_per_year = 365.25 / h

        # guard against degenerate near-zero denominators (e.g. almost every
        # trade landing on an identical breakeven-stop return), which would
        # otherwise blow Sharpe/profit factor up to meaningless +-1e14 values
        std = r.std(ddof=1)
        if pd.notna(std) and std > 1e-6:
            excess = r - rf_annual / periods_per_year
            sharpe = (excess.mean() / std) * np.sqrt(periods_per_year)
        else:
            sharpe = np.nan

        gains, losses = r[r > 0].sum(), -r[r < 0].sum()
        profit_factor = gains / losses if losses > 1e-6 else np.nan

        out.append({
            "days": h,
            "trades": len(r),
            "win_%": round((r > 0).mean() * 100, 2),
            "avg_ret_%": round(r.mean() * 100, 2),
            "cum_ret_%": round(((1 + r * size_pct).prod() - 1) * 100, 2),
            "equity_myr": round(equity.iloc[-1], 2),
            "max_dd_%": round(drawdown.min() * 100, 2),
            "sharpe": round(sharpe, 2) if pd.notna(sharpe) else np.nan,
            "pf": round(profit_factor, 2) if pd.notna(profit_factor) else np.nan,
        })
    return pd.DataFrame(out)


# =============================================================================
# PART 3 -- RUN
# =============================================================================
def main():
    news = load_news(NEWS_CSV_PATH)
    news = filter_trading_hours(news, TRADING_HOURS_START, TRADING_HOURS_END)
    news = filter_headline_quality(news, MIN_HEADLINE_CHARS, MIN_ALPHA_RATIO)
    print(f"[1/5] {len(news)} headlines after trading-hours + quality filters")

    news = score_sentiment_finbert(news, FINBERT_MODEL_NAME, FINBERT_BATCH_SIZE)
    news.to_csv(OUTPUT_DIR / "news_scored.csv", index=False)
    print(f"[2/5] FinBERT scoring done, mean daily score preview computed next")

    daily_signal = aggregate_daily_signal(news, MIN_DAILY_SCORE)
    print(f"[3/5] {len(daily_signal)} trading days had news; "
          f"{daily_signal['go_long'].sum()} qualify as long signals")

    price = load_price(PRICE_PATH)
    print(f"[4/5] {len(price)} price bars, {price['time'].min()} to {price['time'].max()}")

    trades = simulate_trades(daily_signal, price, HORIZONS_DAYS, TRANSACTION_COST_PCT, USE_BREAKEVEN_STOP)
    trades.to_csv(OUTPUT_DIR / "trades_detail.csv", index=False)
    print(f"[5/5] {len(trades)} trades actually entered same-day "
          f"({daily_signal['go_long'].sum() - len(trades)} skipped: no bars left that day)")

    report = performance_report(trades, HORIZONS_DAYS, INITIAL_CAPITAL, POSITION_SIZE_PCT, RISK_FREE_RATE_ANNUAL)
    report.to_csv(OUTPUT_DIR / "performance_report.csv", index=False)

    pd.set_option("display.width", 160)
    print("\n=== PERFORMANCE REPORT (by forward horizon) ===")
    print(report.to_string(index=False))


if __name__ == "__main__":
    main()

[1/5] 74 headlines after trading-hours + quality filters


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 16049.33it/s]


[2/5] FinBERT scoring done, mean daily score preview computed next
[3/5] 57 trading days had news; 23 qualify as long signals
[4/5] 14590 price bars, 09:01:00 to 17:00:00


TypeError: '<=' not supported between instances of 'Timestamp' and 'str'